In [3]:
# =============================================================================
# CELL 1: DEPENDENCIES & GLOBAL CONFIGURATION
# =============================================================================
# This is the single control panel for the entire script.
# Fill in your details here before running any other cell.
# =============================================================================

import smtplib                        # For sending emails via Gmail SMTP
import imaplib                        # For reading emails via Gmail IMAP
import email                          # For building and parsing email messages
from email.mime.multipart import MIMEMultipart   # Allows emails with multiple parts (text + attachment)
from email.mime.text import MIMEText             # For the plain text body of the email
from email.mime.base import MIMEBase             # For attaching files
from email import encoders                       # Encodes file attachments in base64

import pandas as pd                   # For reading and working with the Excel spreadsheet
from docx import Document             # For reading the Word (.docx) assessment file
import os                             # For checking if files exist on disk
import datetime                       # For timestamping log output


# =============================================================================
# --- SECTION A: YOUR GMAIL CREDENTIALS ---
# =============================================================================
# IMPORTANT: Use your Gmail App Password here, NOT your real Gmail password.
# See the Setup Guide above for how to generate one.

GMAIL_ADDRESS  = "YOUR_GMAIL_ADDRESS"      # ← Replace with your Gmail address
GMAIL_APP_PASSWORD = "YOUR_APP_PASSWORD"   # ← Replace with your 16-char App Password


# =============================================================================
# --- SECTION B: FILE PATHS ---
# =============================================================================
# Point these to the actual files on your computer.
# Tip: If the files are in the same folder as this notebook, just use the filename.

QUESTIONS_DOCX_PATH = "assessment_questions.docx"   # Word doc with assessment questions
SCORES_EXCEL_PATH   = "employee_scores.xlsx"         # Excel sheet with names, emails, scores


# =============================================================================
# --- SECTION C: EMAIL CONTENT TEMPLATES ---
# =============================================================================
# Customize the subject lines and body text for each phase here.

# --- Phase 1: Sending Questions ---
QUESTIONS_EMAIL_SUBJECT = "Monthly Assessment — Action Required"

QUESTIONS_EMAIL_BODY = """\
Dear {name},

Please find below your monthly assessment questions.
Kindly complete your responses in a Word document and reply to this email
with your completed file attached by {deadline}.

--- ASSESSMENT QUESTIONS ---
{questions}
----------------------------

Best regards,
The Management Team
"""

QUESTIONS_DEADLINE = "Friday, June 6th, 2026"   # ← Update each month


# --- Phase 3: Sending Scores ---
SCORES_EMAIL_SUBJECT = "Your Monthly Assessment Score"

SCORES_EMAIL_BODY = """\
Dear {name},

Your assessment for this month has been reviewed. Here are your results:

  Score    : {score} / 100
  Feedback : {feedback}

Please reply to this email with the subject line "Score Received — {name}"
to confirm you have received and acknowledged your score.

Best regards,
The Management Team
"""

# --- Phase 4: Acknowledgment Detection ---
# The script will search for emails containing ANY of these keywords in the subject
ACK_KEYWORDS = ["Score Received", "Acknowledgment", "Acknowledged"]


# =============================================================================
# --- SECTION D: EMPLOYEE DISTRIBUTION LIST (Phase 1) ---
# =============================================================================
# This list is used for Phase 1 (sending questions) only.
# For Phase 3, employee data is read directly from the Excel file.
# Format: {"Full Name": "email@address.com"}

EMPLOYEE_LIST = {
    "Jane Doe"    : "jane.doe@company.com",
    "John Smith"  : "john.smith@company.com",
    "Alice Wren"  : "alice.wren@company.com",
    # Add more employees here...
}


# =============================================================================
# --- SECTION E: GMAIL SERVER SETTINGS (Do not change unless necessary) ---
# =============================================================================

SMTP_SERVER = "smtp.gmail.com"   # Gmail's outgoing mail server
SMTP_PORT   = 587                # Port for TLS encryption (standard)
IMAP_SERVER = "imap.gmail.com"   # Gmail's incoming mail server

print("✅ Configuration loaded successfully.")
print(f"   Sender account : {GMAIL_ADDRESS}")
print(f"   Questions file : {QUESTIONS_DOCX_PATH}")
print(f"   Scores file    : {SCORES_EXCEL_PATH}")
print(f"   Employees      : {len(EMPLOYEE_LIST)} on distribution list")


✅ Configuration loaded successfully.
   Sender account : preetibiswal10@gmail.com
   Questions file : assessment_questions.docx
   Scores file    : employee_scores.xlsx
   Employees      : 3 on distribution list


In [4]:
# =============================================================================
# CELL 2: PHASE 1 — WORD DOC PARSER & QUESTION DISTRIBUTION
# =============================================================================
# What this cell does:
#   1. Opens the Word document and extracts all non-empty paragraphs as questions.
#   2. Formats them into a numbered list.
#   3. Sends a personalized email to every employee in EMPLOYEE_LIST.
# =============================================================================


def extract_questions_from_docx(file_path):
    """
    Opens a .docx file and extracts all non-empty paragraphs as a list of strings.
    
    Args:
        file_path (str): Path to the Word document.
    
    Returns:
        list: A list of question strings, or an empty list if the file fails to load.
    """
    # First, verify the file actually exists before trying to open it
    if not os.path.exists(file_path):
        print(f"❌ ERROR: File not found → '{file_path}'")
        print("   Please check the QUESTIONS_DOCX_PATH in Cell 1.")
        return []
    
    try:
        doc = Document(file_path)   # Open the Word document
        questions = []
        
        # Loop through every paragraph in the document
        for paragraph in doc.paragraphs:
            # .strip() removes leading/trailing whitespace
            # We skip blank lines to keep the output clean
            cleaned_text = paragraph.text.strip()
            if cleaned_text:   # Only add if the paragraph is not empty
                questions.append(cleaned_text)
        
        print(f"✅ Extracted {len(questions)} question(s) from '{file_path}'")
        return questions
    
    except Exception as e:
        # Catches any unexpected error (e.g., corrupted file, wrong format)
        print(f"❌ ERROR reading Word document: {e}")
        return []


def format_questions_as_text(questions_list):
    """
    Takes a list of question strings and formats them as a numbered list.
    
    Example output:
        1. What were your key achievements this month?
        2. What challenges did you face?
    
    Args:
        questions_list (list): Raw list of question strings.
    
    Returns:
        str: A single formatted string ready to be inserted into an email body.
    """
    if not questions_list:
        return "No questions found. Please check the Word document."
    
    # Use enumerate to automatically add numbers (starting from 1)
    numbered_lines = [f"{i}. {q}" for i, q in enumerate(questions_list, start=1)]
    
    # Join all lines with a newline character between each
    return "\n".join(numbered_lines)


def send_email(recipient_name, recipient_email, subject, body):
    """
    Sends a single plain-text email using Gmail SMTP.
    
    This function handles the full connection lifecycle:
    connect → authenticate → send → close.
    
    Args:
        recipient_name  (str): Used only for logging/display.
        recipient_email (str): The destination email address.
        subject         (str): Email subject line.
        body            (str): Full plain-text email body.
    
    Returns:
        bool: True if sent successfully, False if an error occurred.
    """
    try:
        # --- Build the email structure ---
        msg = MIMEMultipart()
        msg["From"]    = GMAIL_ADDRESS
        msg["To"]      = recipient_email
        msg["Subject"] = subject
        
        # Attach the body as plain UTF-8 text
        msg.attach(MIMEText(body, "plain", "utf-8"))
        
        # --- Connect to Gmail and send ---
        # smtplib.SMTP opens a connection to the server
        with smtplib.SMTP(SMTP_SERVER, SMTP_PORT) as server:
            server.ehlo()              # Identify ourselves to the server
            server.starttls()          # Upgrade connection to secure TLS
            server.ehlo()              # Re-identify over the secure connection
            server.login(GMAIL_ADDRESS, GMAIL_APP_PASSWORD)   # Authenticate
            server.sendmail(
                from_addr=GMAIL_ADDRESS,
                to_addrs=recipient_email,
                msg=msg.as_string()    # Convert the message object to a raw string
            )
        
        print(f"   ✅ Sent to {recipient_name} <{recipient_email}>")
        return True
    
    except smtplib.SMTPAuthenticationError:
        # This specific error means the credentials are wrong
        print(f"   ❌ AUTHENTICATION FAILED for {recipient_email}")
        print("      → Double-check your GMAIL_ADDRESS and GMAIL_APP_PASSWORD in Cell 1.")
        return False
    
    except smtplib.SMTPException as e:
        # Catches other SMTP-related errors (e.g., connection refused, timeout)
        print(f"   ❌ SMTP error for {recipient_email}: {e}")
        return False
    
    except Exception as e:
        # Catches any other unexpected error
        print(f"   ❌ Unexpected error for {recipient_email}: {e}")
        return False


def run_phase1_send_questions():
    """
    Orchestrates Phase 1: extracts questions from the Word doc and emails
    them to every employee in the EMPLOYEE_LIST dictionary.
    
    Tracks and displays a success/failure summary at the end.
    """
    print("=" * 60)
    print("PHASE 1: SENDING ASSESSMENT QUESTIONS")
    print("=" * 60)
    
    # Step 1: Extract and format questions from the Word doc
    raw_questions = extract_questions_from_docx(QUESTIONS_DOCX_PATH)
    
    if not raw_questions:
        print("⚠️  Halting Phase 1 — no questions to send.")
        return
    
    formatted_questions = format_questions_as_text(raw_questions)
    
    # Step 2: Send to each employee
    print(f"\n📧 Sending to {len(EMPLOYEE_LIST)} employee(s)...\n")
    
    success_count = 0
    failure_count = 0
    
    for name, email_address in EMPLOYEE_LIST.items():
        # Personalize the email body using Python's .format() method
        # The {name}, {questions}, {deadline} placeholders are replaced with real values
        personalized_body = QUESTIONS_EMAIL_BODY.format(
            name      = name,
            questions = formatted_questions,
            deadline  = QUESTIONS_DEADLINE
        )
        
        success = send_email(
            recipient_name  = name,
            recipient_email = email_address,
            subject         = QUESTIONS_EMAIL_SUBJECT,
            body            = personalized_body
        )
        
        if success:
            success_count += 1
        else:
            failure_count += 1
    
    # Step 3: Print a clean summary
    print("\n" + "-" * 40)
    print(f"📊 PHASE 1 COMPLETE")
    print(f"   ✅ Successfully sent : {success_count}")
    print(f"   ❌ Failed            : {failure_count}")
    print("-" * 40)


# --- RUN PHASE 1 ---
# Comment out this line if you want to load the functions without running them yet.
run_phase1_send_questions()


PHASE 1: SENDING ASSESSMENT QUESTIONS
✅ Extracted 3 question(s) from 'assessment_questions.docx'

📧 Sending to 3 employee(s)...

   ✅ Sent to Jane Doe <jane.doe@company.com>
   ✅ Sent to John Smith <john.smith@company.com>
   ✅ Sent to Alice Wren <alice.wren@company.com>

----------------------------------------
📊 PHASE 1 COMPLETE
   ✅ Successfully sent : 3
   ❌ Failed            : 0
----------------------------------------


In [5]:
# CELL 3: PHASE 3 — EXCEL READER & SCORE DISTRIBUTION
# =============================================================================
# What this cell does:
#   1. Reads the Excel spreadsheet (filled in by the manager after reviews).
#   2. Validates that all required columns are present.
#   3. Sends each employee a personalized email with their score and feedback.
#
# PREREQUISITE: The manager must have completed Phase 2 manually —
#   reviewed all submissions and filled in the 'Score' (and optionally
#   'Feedback') columns in the Excel file before running this cell.
# =============================================================================


def load_scores_from_excel(file_path):
    """
    Reads the employee scores Excel file into a pandas DataFrame.
    
    Validates that the required columns exist. The 'Feedback' column
    is optional — if absent, a default message will be used.
    
    Args:
        file_path (str): Path to the Excel file.
    
    Returns:
        pd.DataFrame or None: The loaded data, or None if loading fails.
    """
    # Check that the file actually exists
    if not os.path.exists(file_path):
        print(f"❌ ERROR: File not found → '{file_path}'")
        print("   Please check the SCORES_EXCEL_PATH in Cell 1.")
        return None
    
    try:
        # Read the Excel file — pandas handles all the heavy lifting here
        df = pd.read_excel(file_path, engine="openpyxl")
        
        print(f"✅ Loaded '{file_path}' — {len(df)} row(s) found.")
        
        # --- Validate required columns ---
        # These two columns MUST be present for the script to work
        required_columns = ["Name", "Email", "Score"]
        missing_columns = [col for col in required_columns if col not in df.columns]
        
        if missing_columns:
            print(f"❌ ERROR: Missing required column(s): {missing_columns}")
            print("   Please check your Excel file matches the template in the Setup Guide.")
            return None
        
        # Add a 'Feedback' column with a default value if it doesn't exist
        if "Feedback" not in df.columns:
            print("ℹ️  No 'Feedback' column found — using default message for all employees.")
            df["Feedback"] = "No additional feedback provided."
        else:
            # Fill any empty/NaN feedback cells with a default message
            df["Feedback"] = df["Feedback"].fillna("No additional feedback provided.")
        
        # Remove any completely empty rows (a common Excel issue)
        df = df.dropna(subset=["Name", "Email", "Score"])
        
        print(f"   Rows ready to process: {len(df)}")
        return df
    
    except Exception as e:
        print(f"❌ ERROR reading Excel file: {e}")
        return None


def validate_score_row(row):
    """
    Checks a single row from the DataFrame for data quality issues.
    
    Args:
        row (pd.Series): A single row from the scores DataFrame.
    
    Returns:
        tuple: (is_valid: bool, error_message: str)
    """
    # Check that the score is a number
    try:
        score = float(row["Score"])
    except (ValueError, TypeError):
        return False, f"Score '{row['Score']}' is not a valid number."
    
    # Check that the score is within a sensible range
    if not (0 <= score <= 100):
        return False, f"Score {score} is outside the expected 0–100 range."
    
    # Check that the email address looks valid (very basic check)
    if "@" not in str(row["Email"]):
        return False, f"Email '{row['Email']}' does not look valid."
    
    return True, ""   # All checks passed


def run_phase3_send_scores():
    """
    Orchestrates Phase 3: reads the Excel file and emails each employee
    their individual score and feedback.
    """
    print("=" * 60)
    print("PHASE 3: SENDING INDIVIDUAL SCORES")
    print("=" * 60)
    
    # Step 1: Load the spreadsheet
    scores_df = load_scores_from_excel(SCORES_EXCEL_PATH)
    
    if scores_df is None:
        print("⚠️  Halting Phase 3 — could not load scores data.")
        return
    
    # Step 2: Preview what will be sent (helps catch mistakes before sending)
    print("\n📋 Preview of data to be sent:")
    # Display only Name, Email, and Score columns for a clean preview
    print(scores_df[["Name", "Email", "Score"]].to_string(index=False))
    print()
    
    # Step 3: Loop through each row and send emails
    print(f"📧 Sending scores to {len(scores_df)} employee(s)...\n")
    
    success_count = 0
    failure_count = 0
    skipped_count = 0
    
    # iterrows() lets us loop through a DataFrame row by row
    for index, row in scores_df.iterrows():
        name  = str(row["Name"]).strip()
        email_address = str(row["Email"]).strip()
        score = row["Score"]
        feedback = str(row["Feedback"]).strip()
        
        # Validate this row before trying to send
        is_valid, error_msg = validate_score_row(row)
        if not is_valid:
            print(f"   ⚠️  Skipping {name}: {error_msg}")
            skipped_count += 1
            continue   # Move on to the next row
        
        # Format the score nicely — show as integer if it's a whole number
        score_display = int(score) if float(score) == int(float(score)) else score
        
        # Personalize the email body for this specific employee
        personalized_body = SCORES_EMAIL_BODY.format(
            name     = name,
            score    = score_display,
            feedback = feedback
        )
        
        success = send_email(
            recipient_name  = name,
            recipient_email = email_address,
            subject         = SCORES_EMAIL_SUBJECT,
            body            = personalized_body
        )
        
        if success:
            success_count += 1
        else:
            failure_count += 1
    
    # Step 4: Final summary
    print("\n" + "-" * 40)
    print(f"📊 PHASE 3 COMPLETE")
    print(f"   ✅ Successfully sent : {success_count}")
    print(f"   ❌ Failed            : {failure_count}")
    print(f"   ⚠️  Skipped (bad data): {skipped_count}")
    print("-" * 40)


# --- RUN PHASE 3 ---
run_phase3_send_scores()


PHASE 3: SENDING INDIVIDUAL SCORES
✅ Loaded 'employee_scores.xlsx' — 2 row(s) found.
   Rows ready to process: 2

📋 Preview of data to be sent:
      Name            Email  Score
  John Doe john@example.com     85
Jane Smith jane@example.com     92

📧 Sending scores to 2 employee(s)...

   ✅ Sent to John Doe <john@example.com>
   ✅ Sent to Jane Smith <jane@example.com>

----------------------------------------
📊 PHASE 3 COMPLETE
   ✅ Successfully sent : 2
   ❌ Failed            : 0
   ⚠️  Skipped (bad data): 0
----------------------------------------


In [6]:
# CELL 4: PHASE 4 — ACKNOWLEDGMENT CHECKER
# =============================================================================
# What this cell does:
#   1. Connects to Gmail via IMAP (read-only access to your inbox).
#   2. Searches for UNREAD emails containing acknowledgment keywords in
#      the subject line, received within the last N days.
#   3. Prints a clear summary of who has and hasn't acknowledged yet,
#      cross-referenced against the Excel scores file.
# =============================================================================


def fetch_acknowledgment_replies(days_back=2):
    """
    Connects to Gmail via IMAP and retrieves unread emails whose subject
    line contains any of the keywords defined in ACK_KEYWORDS.
    
    Args:
        days_back (int): How many days back to search. Defaults to 7.
    
    Returns:
        list of dict: Each dict contains 'sender', 'subject', and 'date'
                      for a matched email. Returns an empty list on failure.
    """
    print(f"🔍 Scanning inbox for acknowledgment emails (last {days_back} days)...")
    
    found_emails = []
    
    try:
        # --- Connect to Gmail's IMAP server ---
        mail = imaplib.IMAP4_SSL(IMAP_SERVER)   # SSL for secure connection
        mail.login(GMAIL_ADDRESS, GMAIL_APP_PASSWORD)
        
        # Select the INBOX folder (read-only=False means we can mark as read if needed)
        mail.select("inbox", readonly=True)
        
        # --- Build the date filter ---
        # IMAP uses "DD-Mon-YYYY" date format (e.g., "25-May-2026")
        since_date = (datetime.datetime.now() - datetime.timedelta(days=days_back))
        since_str  = since_date.strftime("%d-%b-%Y")
        
        # --- Search for matching emails ---
        # We search for UNSEEN (unread) emails since our cutoff date.
        # Note: IMAP SEARCH cannot filter by subject keywords natively in all
        # servers, so we'll fetch all recent unread emails and filter ourselves.
        
        search_criteria = f'(UNSEEN SINCE "{since_str}")'
        status, message_ids = mail.search(None, search_criteria)
        
        if status != "OK" or not message_ids[0]:
            print("   ℹ️  No new unread emails found in the specified date range.")
            mail.logout()
            return []
        
        # message_ids[0] is a space-separated byte string of IDs, e.g. b"1 4 7 12"
        id_list = message_ids[0].split()
        print(f"   Found {len(id_list)} unread email(s) in range. Filtering by keyword...")
        
        # --- Filter by acknowledgment keywords ---
        for msg_id in id_list:
            # Fetch only the headers (RFC822.HEADER) to avoid downloading full bodies
            status, msg_data = mail.fetch(msg_id, "(RFC822.HEADER)")
            
            if status != "OK":
                continue   # Skip this message if fetch failed
            
            # Parse the raw header bytes into an email object
            raw_header = msg_data[0][1]
            parsed_msg = email.message_from_bytes(raw_header)
            
            subject = parsed_msg.get("Subject", "")   # Get subject, default to empty string
            sender  = parsed_msg.get("From",    "")
            date    = parsed_msg.get("Date",    "")
            
            # Check if any of our ACK_KEYWORDS appear in the subject (case-insensitive)
            subject_lower = subject.lower()
            keyword_matched = any(kw.lower() in subject_lower for kw in ACK_KEYWORDS)
            
            if keyword_matched:
                found_emails.append({
                    "sender"  : sender,
                    "subject" : subject,
                    "date"    : date
                })
        
        mail.logout()   # Always close the connection cleanly
        
    except imaplib.IMAP4.error as e:
        # IMAP-specific errors (wrong credentials, server issue, etc.)
        print(f"❌ IMAP ERROR: {e}")
        print("   → Check your GMAIL_ADDRESS and GMAIL_APP_PASSWORD in Cell 1.")
        print("   → Make sure IMAP is enabled in Gmail: Settings → See all settings → Forwarding and POP/IMAP")
    
    except Exception as e:
        print(f"❌ Unexpected error during inbox scan: {e}")
    
    return found_emails


def extract_email_address(raw_sender_string):
    """
    Extracts just the email address from a raw 'From' header string.
    
    Example:
        Input : "Jane Doe <jane.doe@company.com>"
        Output: "jane.doe@company.com"
    
    Args:
        raw_sender_string (str): The raw 'From' field from an email header.
    
    Returns:
        str: The lowercase email address.
    """
    # email.utils.parseaddr handles both "Name <email>" and bare "email" formats
    _, addr = email.utils.parseaddr(raw_sender_string)
    return addr.lower().strip()


def run_phase4_check_acknowledgments(days_back=2
):
    """
    Orchestrates Phase 4: fetches acknowledgment emails and cross-references
    them with the employee list from the Excel file to show who has and
    hasn't responded yet.
    
    Args:
        days_back (int): How many days back to look for acknowledgment emails.
    """
    print("=" * 60)
    print("PHASE 4: ACKNOWLEDGMENT TRACKER")
    print("=" * 60)
    
    # Step 1: Fetch acknowledgment emails from inbox
    ack_emails = fetch_acknowledgment_replies(days_back=days_back)
    
    # Build a set of email addresses that have sent acknowledgments
    # (sets allow fast "is X in this group?" lookups)
    ack_senders = set()
    
    if ack_emails:
        print(f"\n📬 {len(ack_emails)} acknowledgment email(s) detected:\n")
        for ack in ack_emails:
            addr = extract_email_address(ack["sender"])
            ack_senders.add(addr)
            # Truncate long subjects for cleaner display
            short_subject = (ack["subject"][:55] + "...") if len(ack["subject"]) > 55 else ack["subject"]
            print(f"   ✅ From    : {ack['sender']}")
            print(f"      Subject : {short_subject}")
            print(f"      Date    : {ack['date']}")
            print()
    else:
        print("   ℹ️  No matching acknowledgment emails found.")
    
    # Step 2: Cross-reference with the employee scores list
    print("-" * 60)
    print("📋 EMPLOYEE ACKNOWLEDGMENT STATUS")
    print("-" * 60)
    
    # Try to load the scores Excel to get the full employee list
    if os.path.exists(SCORES_EXCEL_PATH):
        try:
            df = pd.read_excel(SCORES_EXCEL_PATH, engine="openpyxl")
            
            if "Name" in df.columns and "Email" in df.columns:
                acknowledged     = []
                not_acknowledged = []
                
                for _, row in df.iterrows():
                    name  = str(row["Name"]).strip()
                    email_address = str(row["Email"]).strip().lower()
                    
                    if email_address in ack_senders:
                        acknowledged.append(name)
                    else:
                        not_acknowledged.append(name)
                
                # Print acknowledged employees
                print(f"\n✅ Acknowledged ({len(acknowledged)}):")
                if acknowledged:
                    for name in acknowledged:
                        print(f"   • {name}")
                else:
                    print("   — None yet")
                
                # Print employees who haven't responded
                print(f"\n⏳ Awaiting Acknowledgment ({len(not_acknowledged)}):")
                if not_acknowledged:
                    for name in not_acknowledged:
                        print(f"   • {name}")
                else:
                    print("   — Everyone has responded! 🎉")
            else:
                print("⚠️  Excel file found but missing 'Name' or 'Email' columns.")
        
        except Exception as e:
            print(f"⚠️  Could not read Excel for cross-reference: {e}")
    
    else:
        # Fallback: just show raw acknowledgment senders without cross-referencing
        print("ℹ️  (Excel scores file not found — showing raw senders only)")
        if ack_senders:
            for addr in sorted(ack_senders):
                print(f"   ✅ {addr}")
        else:
            print("   No acknowledgments received yet.")
    
    print("\n" + "=" * 60)
    print(f"Scan complete — {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 60)


# --- RUN PHASE 4 ---
# Change days_back to widen or narrow the search window
run_phase4_check_acknowledgments(days_back=2)


PHASE 4: ACKNOWLEDGMENT TRACKER
🔍 Scanning inbox for acknowledgment emails (last 2 days)...
   Found 64 unread email(s) in range. Filtering by keyword...
   ℹ️  No matching acknowledgment emails found.
------------------------------------------------------------
📋 EMPLOYEE ACKNOWLEDGMENT STATUS
------------------------------------------------------------

✅ Acknowledged (0):
   — None yet

⏳ Awaiting Acknowledgment (2):
   • John Doe
   • Jane Smith

Scan complete — 2026-06-05 18:50:53
